# 05 — Artificial Neural Network (ANN) Architecture & Training
### Household Electricity Consumption Prediction Using ANN
**Author:** Roshan Langhi | B.Tech Computer Engineering

---
### 📌 Objective
Construct a deep neural network using TensorFlow/Keras with Dropout and Batch Normalization, compile with Adam/MSE, and train the model.



In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

PROCESSED_DATA_PATH = os.path.join("..", "data", "processed", "cleaned_data.csv")
MODEL_PATH = os.path.join("..", "models", "electricity_ann.keras")
SCALER_PATH = os.path.join("..", "models", "scaler.pkl")



### 1. Load Data and Fitted Scaler


In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
feature_cols = [
    'Global_reactive_power', 'Voltage', 'Global_intensity',
    'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',
    'Hour', 'Day', 'Month', 'DayOfWeek'
]
target_col = 'Global_active_power'

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = joblib.load(SCALER_PATH)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)



### 2. Construct ANN Architecture
- **Layer 1:** Dense 64 units, ReLU activation
- **Batch Normalization:** Stabilizes training distribution
- **Dropout (0.2):** Regularization to prevent overfitting
- **Layer 2:** Dense 32 units, ReLU activation
- **Dropout (0.2):** Regularization
- **Layer 3:** Dense 16 units, ReLU activation
- **Output Layer:** Dense 1 unit, Linear activation (regression)



In [ ]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='linear')
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()



### 3. Model Training with Early Stopping


In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)



### 4. Plot Training vs Validation Loss Curve


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history.history['loss'], label='Training Loss (MSE)', color='#2563EB', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)', color='#EF4444', linewidth=2)
plt.title("ANN Model Training vs Validation Loss", fontsize=14, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.tight_layout()
plt.show()



### 5. Save Trained Model


In [ ]:
model.save(MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

